# La DataFrame et la Série

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from _data import load_penguins

## 1. Une DataFrame, c'est quoi ?

Une DataFrame est un **dictionnaire de colonnes**, où chaque colonne est une Series,
et toutes les colonnes partagent le même Index.

```
         nom      age   ville
       ┌────────┬─────┬────────┐
  0    │ Alice  │  32 │ Paris  │
  1    │ Bob    │  28 │ Lyon   │
  2    │ Claire │  35 │ Nantes │
  3    │ David  │  41 │ Paris  │
       └────────┴─────┴────────┘
  ▲         ▲
Index    Colonnes (chacune est une Series)
```

On peut créer une DataFrame à partir d'un dictionnaire — les clés deviennent les noms de colonnes :

In [ ]:
df = pd.DataFrame({
    "nom":   ["Alice", "Bob", "Claire", "David"],
    "age":   [32, 28, 35, 41],
    "ville": ["Paris", "Lyon", "Nantes", "Paris"],
})
df

## 2. Series

Une Series, c'est une colonne : un tableau de valeurs avec un Index et un nom.

In [ ]:
s = df["age"]
s

In [ ]:
print("nom   :", s.name)
print("dtype :", s.dtype)
print("index :", s.index.tolist())

## 3. Index

L'Index est la colonne de labels qui identifie chaque ligne.
Par défaut c'est un entier (0, 1, 2…), mais ça peut être n'importe quoi.

Son rôle principal : **aligner automatiquement les valeurs** lors des opérations entre Series.

Avant d'exécuter la cellule suivante — **que va produire `s1 + s2` à votre avis ?**

In [ ]:
# Deux Series avec des index différents
s1 = pd.Series([10, 20, 30], index=["a", "b", "c"])
s2 = pd.Series([100, 200, 300], index=["b", "c", "d"])

s1 + s2

**Ce qui se passe** : pandas aligne les deux Series sur leur index commun (`b`, `c`),
et produit `NaN` pour les labels qui n'existent que d'un côté (`a`, `d`).

Ce comportement est la raison d'être de l'Index. C'est aussi la source du bug vu dans le notebook 9
(`.assign()` sans `lambda`) : quand deux objets n'ont pas le même index, l'alignement produit des `NaN`
silencieusement.

> **À retenir** : l'Index n'est pas juste un numéro de ligne. C'est la clé d'alignement de toutes les opérations pandas.

## 4. Arithmétique vectorisée

Les opérations sur une Series s'appliquent à **toutes les valeurs d'un coup**, sans boucle Python.
C'est ce qu'on appelle l'arithmétique vectorisée.

In [ ]:
penguins = load_penguins()
penguins.head(3)

In [ ]:
# Ratio longueur de bec / masse corporelle — une opération, toutes les lignes
(penguins["bill_length_mm"] / penguins["body_mass_g"]).head()

In [ ]:
# Comparaison de vitesse : boucle Python vs vectorisé
data = penguins["body_mass_g"].dropna()

In [ ]:
%%timeit
# Boucle Python
result = []
for val in data:
    result.append(val / 1000)

In [ ]:
%%timeit
# Vectorisé
data / 1000

La version vectorisée est typiquement **10 à 100 fois plus rapide** sur un vrai dataset.
Elle est aussi plus courte et plus lisible.

Règle pratique : si vous vous retrouvez à écrire une boucle `for` sur les lignes d'une DataFrame,
il existe presque toujours une opération vectorisée équivalente. Le notebook 5 (`apply / map / transform`)
couvre les cas où ce n'est pas évident.

## 5. DataFrame vs NumPy array

Sous le capot, une DataFrame stocke ses données dans des tableaux NumPy.
La différence clé :

In [ ]:
arr = penguins[["bill_length_mm", "body_mass_g"]].dropna().to_numpy()
print(type(arr))
print(arr[:3])

| | DataFrame / Series | NumPy array |
|---|---|---|
| Index | Oui — alignement automatique | Non |
| Noms de colonnes | Oui | Non |
| Colonnes de types mixtes | Oui (chaque colonne a son dtype) | Non (un seul dtype pour tout le tableau) |
| Valeurs manquantes | Oui (`NaN`, `pd.NA`) | Partiel (`np.nan` float uniquement) |

Utilisez `.to_numpy()` pour passer à NumPy quand une librairie l'exige (scikit-learn, SciPy…).
Évitez `.values` — c'est l'ancienne API, toujours fonctionnelle mais moins explicite.

## Quiz

<details><summary>Question 1 — <code>df["age"]</code> et <code>df[["age"]]</code> ont-ils le même type ?</summary>

Non. `df["age"]` est une **Series**, `df[["age"]]` est une **DataFrame** à une colonne.
Même résultat visuel dans un notebook, mais type différent — et certaines fonctions
acceptent l'un et pas l'autre.

</details>

<details><summary>Question 2 — Pourquoi <code>s1 + s2</code> produit-il des NaN quand les index ne se recoupent pas entièrement ?</summary>

Parce que pandas aligne les opérations sur l'index avant de calculer.
Pour un label présent dans une seule des deux Series, il n'y a pas de valeur à additionner de l'autre côté :
le résultat est `NaN`. C'est un choix de conception délibéré — il vaut mieux un `NaN` visible
qu'un résultat silencieusement incorrect.

</details>

<details><summary>Question 3 — Quand utiliser <code>.to_numpy()</code> ?</summary>

Quand vous passez des données à une librairie qui n'accepte pas de DataFrame/Series en entrée :
scikit-learn, SciPy, certaines fonctions matplotlib. Dans le reste du code pandas, gardez vos
Series et DataFrames — la conversion fait perdre l'index et les noms de colonnes.

</details>